In [1]:
import sys
import os

# Go up one level to the main project directory and add it to Python's path
sys.path.append(os.path.abspath(os.path.join(os.getcwd(), '..')))

In [2]:
from xgboost import XGBClassifier
from sklearn.metrics import classification_report, confusion_matrix
from src.preprocess import preprocess_data
import pandas as pd

In [3]:
train_data = pd.read_csv(r"..\data\raw\train.csv")

In [4]:
df = preprocess_data(train_data)

In [5]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report
from xgboost import XGBClassifier
from src.preprocess import preprocess_data # Using your newly updated function!

# ==========================================
# PHASE 1: LOCAL VALIDATION
# ==========================================
print("--- PHASE 1: Local Validation ---")

# 1. Separate Features and Target from your cleaned training data
X = df.drop('health_condition', axis=1)
y = df['health_condition']

# 2. Split into Local Train/Test
X_train_local, X_test_local, y_train_local, y_test_local = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# 3. Initialize and Train Local Model
local_model = XGBClassifier(random_state=42, n_jobs=-1, eval_metric='logloss')
print("Training local model...")
local_model.fit(X_train_local, y_train_local)

# 4. Evaluate Locally
local_preds = local_model.predict(X_test_local)
print("Local Validation Score:\n")
print(classification_report(y_test_local, local_preds))

--- PHASE 1: Local Validation ---
Training local model...
Local Validation Score:

              precision    recall  f1-score   support

           0       0.97      0.99      0.98    118512
           1       0.95      0.80      0.87      7961
           2       0.96      0.78      0.86     11545

    accuracy                           0.97    138018
   macro avg       0.96      0.86      0.90    138018
weighted avg       0.97      0.97      0.96    138018



In [7]:
# ==========================================
# PHASE 2: KAGGLE SUBMISSION
# ==========================================
print("\n--- PHASE 2: Kaggle Submission ---")

# 1. Retrain on 100% of the Training Data for maximum accuracy
final_model = XGBClassifier(random_state=42, n_jobs=-1, eval_metric='logloss')
print("Retraining model on ALL training data...")
final_model.fit(X, y)

# 2. Load and clean Kaggle's test.csv
raw_test_df = pd.read_csv(r'..\data\raw\test.csv') 
passenger_ids = raw_test_df['id'] # Save IDs for submission!

# Apply your leak-free preprocess function
clean_test_df = preprocess_data(raw_test_df) 

# Align columns perfectly with training data
X_test_kaggle = clean_test_df.reindex(columns=X.columns, fill_value=0)

# 3. Make Final Predictions
print("Generating predictions for Kaggle...")
kaggle_preds = final_model.predict(X_test_kaggle)

# 4. Format and Save Submission
submission = pd.DataFrame({
    'id': passenger_ids,
    'health_condition': kaggle_preds
})

# Optional: Map numeric predictions (0,1,2) back to text labels if Kaggle requires text!
reverse_mapping = {0: 'at-risk', 1: 'fit', 2: 'unhealthy'}
submission['health_condition'] = submission['health_condition'].map(reverse_mapping)

submission.to_csv('submission.csv', index=False)
print("submission.csv successfully created! Ready for upload.")


--- PHASE 2: Kaggle Submission ---
Retraining model on ALL training data...
Generating predictions for Kaggle...
submission.csv successfully created! Ready for upload.


In [8]:
submission.shape

(295753, 2)